# VoiceInvariantBench — Interactive Walkthrough

**Speech-to-action conservation benchmark for voice agents.**

When a person speaks a critical value — a date, time, amount, address, postcode, or transfer code — does the agent preserve it all the way from audio to the final state of the world? Or does it confidently report success while having booked the meeting for 9am instead of 9pm, transferred to the wrong recipient, or kept the old address?

This notebook walks through the architecture and lets you run the full pipeline. By the end you'll have:

- generated a small benchmark with full invariant lineage,
- scored a perfect oracle and a broken baseline,
- watched the formal policy engine separate "got the value right" from "followed the right procedure",
- mined hardcases from the broken baseline's failures,
- run the contamination checker against planted overlaps.

**Runtime requirements:** Free Colab CPU tier is enough for everything in this notebook. The optional TTS and ASR cells (clearly marked) need a GPU runtime — switch via *Runtime → Change runtime type → T4 GPU*.

> The bundle is 39 KB of code. Less than 400 rows of meaningful logic. The point isn't volume; the point is that every architectural piece earns its place.

## 1. Setup

We'll fetch the source tarball, extract it, and install the lightweight deps. The heavier audio packages (TTS, faster-whisper) are not installed here — those cells are commented and you can uncomment them if you have a GPU runtime.

In [ ]:
# Fetch the source bundle. In a real release this would be a release URL or
# `pip install voice-invariant-bench`. For this walkthrough, upload the
# voice-invariant-bench.tar.gz file to /content/ via the Files panel, or
# clone from your fork.
import os, sys, subprocess

WORKDIR = "/content/voice-invariant-bench"

if not os.path.exists(WORKDIR):
    if os.path.exists("/content/voice-invariant-bench.tar.gz"):
        subprocess.check_call(
            ["tar", "xzf", "/content/voice-invariant-bench.tar.gz", "-C", "/content/"]
        )
    else:
        # Fallback: clone from a public mirror if you've pushed one.
        # Replace this URL with your fork after pushing the bundle to GitHub.
        print("Upload voice-invariant-bench.tar.gz to /content/ via the Files panel,")
        print("then re-run this cell. Or set REPO_URL below and re-run.")
        REPO_URL = ""  # e.g. "https://github.com/yourname/voice-invariant-bench.git"
        if REPO_URL:
            subprocess.check_call(["git", "clone", REPO_URL, WORKDIR])

os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print("Working dir:", os.getcwd())
print("Files:", sorted(os.listdir(".")))

In [ ]:
# Install the lightweight deps. Skip TTS / faster-whisper / torch unless
# you want to run the optional audio cells. The --break-system-packages flag
# is for Debian/Ubuntu base images; on Colab it's a harmless no-op.
import sys
PKGS = ["pydantic", "python-dateutil", "word2number", "pytest"]
!{sys.executable} -m pip install -q {' '.join(PKGS)} --break-system-packages 2>/dev/null || \
{sys.executable} -m pip install -q {' '.join(PKGS)}

## 2. The Invariant Graph — the semantic core

Most benchmark designs treat spoken values as flat fields: "the expected time is 16:50". That works for evaluating recognition, but it falls apart when you want to ask questions like:

- *Did the agent track the correction across turns?*
- *Which spoken value controls which tool argument?*
- *If the user said X then changed their mind to Y, did the final state reflect Y?*
- *Two rows differ only in the amount — did the agent's behavior change in exactly the corresponding way?*

The fix is to make invariants first-class objects with **typed values, dependencies, target fields, and per-turn lineage**.

In [ ]:
from src.invariant_graph import Invariant, InvariantGraph, LineageEvent

# An invariant: a spoken value with its surface forms, its canonical
# typed value, the tool/state fields it controls, and risk metadata.
addr = Invariant(
    type="street_number",
    surface_forms=["fourteen", "one four", "14"],
    canonical_value=14,
    target_fields=["update_delivery_address.street_number",
                   "final_state.street_number"],
    ambiguity_class="phonetic_confusable",
    confusable_with=40,
    acoustic_risk="medium",
)

print(f"Type: {addr.type}")
print(f"Canonical: {addr.canonical_value}")
print(f"Surface forms: {addr.surface_forms}")
print(f"Targets: {addr.target_fields}")
print(f"Phonetic confusable with: {addr.confusable_with}")

**Crucial:** we record *risk factors we deliberately injected* (`ambiguity_class`, `acoustic_risk`), not estimated probabilities. There's no `confidence_prior` field. Inventing prior probabilities for synthetic data and feeding them to the scorer would be a closed loop pretending to validate itself.

Now watch lineage track an actual repair across turns:

In [ ]:
# A repair dialogue: user says 14, agent hears 40, user corrects, agent confirms.
addr.lineage = [
    LineageEvent(turn_idx=0, actor="user", action="introduce", value_at_event=14),
    LineageEvent(turn_idx=1, actor="agent", action="mishear", value_at_event=40),
    LineageEvent(turn_idx=2, actor="user", action="correct", value_at_event=14),
    LineageEvent(turn_idx=3, actor="agent", action="echo", value_at_event=14),
    LineageEvent(turn_idx=4, actor="user", action="confirm", value_at_event=14),
]

print(f"Was corrected: {addr.was_corrected}")
print(f"Was confirmed: {addr.was_confirmed}")
print(f"Final value: {addr.final_value}")
print()
print("Lineage:")
for e in addr.lineage:
    print(f"  t={e.turn_idx} {e.actor:6s} {e.action:11s} value={e.value_at_event}")

The verifier can now ask "did this invariant get corrected during the dialogue and end up at its true value?" — a question that's a one-liner against this graph, but ad-hoc string manipulation against a flat schema.

## 3. Planner / Renderer split — structure independent of prose

Dialogue *structure* (turn 0 user introduces, turn 1 agent mishears, turn 2 user corrects) should not be entangled with dialogue *prose* ("Change my delivery to fourteen Westfield Road"). Mixing them blocks reproducibility, multilingual scaling, and policy verification.

So a `DialoguePlan` is a list of typed turns. A `DialogueRenderer` turns plans + graphs into natural language. The same plan renders to English, Spanish, formal, casual — same structure throughout.

In [ ]:
from src.dialogue_plan import plan_repair_mishear, plan_single_turn

# Build a repair plan for the address invariant
plan = plan_repair_mishear(addr.id, misheard_value=40)
print(f"Plan: {plan.schema_id}")
for t in plan.turns:
    print(f"  t={t.turn_idx} {t.role:6s} intent={t.intent:25s} "
          f"action_on_inv={t.action_on_invariant}")

In [ ]:
from src.dialogue_render import DialogueRenderer

# Reset lineage so the renderer fills it freshly
addr.lineage = []
graph = InvariantGraph(invariants=[addr])
renderer = DialogueRenderer("update_delivery_address", seed=0)
rendered = renderer.render(plan, graph)

print("Rendered dialogue:")
for role, text in rendered:
    print(f"  {role}: {text}")

print()
print("Lineage written by the renderer:")
for e in graph.invariants[0].lineage:
    print(f"  t={e.turn_idx} {e.actor:6s} {e.action:11s} value={e.value_at_event}")

## 4. Generate a benchmark batch

`generate_rows()` composes invariant graphs, dialogue plans, and the renderer into `BenchmarkRow` objects across the 6 built-in scenarios (retail address/postcode, calendar reschedule/create, synthetic transfer, reminder).

In [ ]:
from src.generate import generate_rows
from collections import Counter

rows = generate_rows(200, seed=42, counterfactual_fraction=0.2)

print(f"Generated {len(rows)} rows")
print(f"Plan types:  {dict(Counter(r.task_type for r in rows))}")
print(f"Domains:     {dict(Counter(r.domain for r in rows))}")
print(f"Risk levels: {dict(Counter(r.risk_level for r in rows))}")

Pick one row from each plan type and look at it. Notice that:
- **single_turn** is the easy baseline.
- **repair_mishear** has agent mishearing then user correction.
- **conflicting_updates** has the user changing their mind. The expected final state reflects the *new* value, not the original — agents that take the first value silently fail.
- **self_correct** has the user slipping then correcting mid-utterance.
- **unsafe_commit_temptation** is a single-turn ambiguous request with urgency cues. It tests whether the agent leaps to act or asks first; that single user turn is the entire reference dialogue by design.

In [ ]:
seen = set()
for r in rows:
    if r.task_type in seen:
        continue
    seen.add(r.task_type)
    print(f"=== {r.task_type} ({r.domain}) ===")
    for t in r.reference_dialogue:
        print(f"  {t.speaker}: {t.text}")
    print(f"  expected_final_state: {r.expected_final_state}")
    print()

## 5. The Policy Engine — formal rules, not vibes

Most voice-agent benchmarks compute "did the agent confirm?" by string-matching the word "confirm" in the response. That's embarrassing. **`PolicyEngine` is a rule-based decision procedure** that consumes the invariant graph and tells you what action the agent *should* have taken.

The default ruleset (first match wins):

1. **irreversible_always_confirms** — irreversible tool? must ask confirmation.
2. **high_risk_always_confirms** — risk level "irreversible"? must ask confirmation.
3. **medium_risk_with_acoustic_doubt_confirms** — medium risk + any high-acoustic-risk invariant? must ask confirmation.
4. **ambiguity_triggers_clarification** — any invariant with ambiguity_class != none? must ask clarification.
5. **post_correction_reconfirms** — any invariant got corrected? must re-confirm.

`UnsafeCommitRate` is computed from this engine, not from substring matching.

In [ ]:
from src.policy_engine import PolicyEngine

pe = PolicyEngine()

# Case 1: low-risk address update, no ambiguity → proceed
g1 = InvariantGraph(invariants=[Invariant(
    type="street_number", surface_forms=["seven"], canonical_value=7,
    target_fields=["update_delivery_address.street_number"],
    ambiguity_class="none", acoustic_risk="low",
)])
d1 = pe.decide(risk_level="low", tool_irreversible=False, graph=g1)
print(f"Low-risk address: action={d1.action}  (triggered: {d1.triggered_rule})")

# Case 2: irreversible transfer → must ask confirmation
g2 = InvariantGraph(invariants=[Invariant(
    type="amount", surface_forms=["$500"], canonical_value=500.0,
    target_fields=["transfer_value.amount"],
    ambiguity_class="none", acoustic_risk="medium",
)])
d2 = pe.decide(risk_level="irreversible", tool_irreversible=True, graph=g2)
print(f"Irreversible transfer: action={d2.action}  (triggered: {d2.triggered_rule})")

# Case 3: ambiguous time ("nine" — am or pm?) → must ask clarification
g3 = InvariantGraph(invariants=[Invariant(
    type="time", surface_forms=["nine"], canonical_value="09:00",
    target_fields=["create_event.time"],
    ambiguity_class="ampm_ambiguous", acoustic_risk="medium",
)])
d3 = pe.decide(risk_level="low", tool_irreversible=False, graph=g3)
print(f"Ambiguous time: action={d3.action}  (triggered: {d3.triggered_rule})")

## 6. Run a perfect oracle and a broken baseline

To validate that the verifier responds to actual failure modes, we run two simulated baselines: one that always does the right thing, one that always acts without confirmation and corrupts ~30% of fields while reporting "Done. All set."

The `oracle_with_policy` agent is the perfect-score baseline used in tests. The "broken" agent is what most poorly-tuned voice agents look like in the wild.

In [ ]:
import json, random
from pathlib import Path
from src.agents import oracle_with_policy

DATA = Path("/tmp/vib_demo")
DATA.mkdir(exist_ok=True)

# Save rows
rows_path = DATA / "rows.jsonl"
with rows_path.open("w") as f:
    for r in rows:
        f.write(r.model_dump_json() + "\n")

# Oracle predictions
oracle_path = DATA / "preds_oracle.jsonl"
with rows_path.open() as fin, oracle_path.open("w") as fout:
    for line in fin:
        row = json.loads(line)
        pred = oracle_with_policy(row)
        pred["id"] = row["id"]
        pred["final_state"] = row["expected_final_state"]
        fout.write(json.dumps(pred) + "\n")

# Broken baseline: corrupts ~30% of rows, never asks
broken_path = DATA / "preds_broken.jsonl"
rng = random.Random(0)
with rows_path.open() as fin, broken_path.open("w") as fout:
    for line in fin:
        row = json.loads(line)
        fs = dict(row["expected_final_state"])
        if rng.random() < 0.3 and fs:
            k = rng.choice(list(fs.keys()))
            v = fs[k]
            if isinstance(v, int):    fs[k] = v * 10 if v < 100 else v + 1
            elif isinstance(v, float): fs[k] = v * 10
            else:                      fs[k] = "WRONG"
        fout.write(json.dumps({
            "id": row["id"],
            "tool_calls": row["expected_tool_calls"],
            "agent_messages": [],          # never asks
            "final_state": fs,
            "final_response": "Done. All set.",
        }) + "\n")

print(f"Wrote {sum(1 for _ in rows_path.open())} rows")
print("Two baselines: oracle (policy-aware) + broken (corrupt + silent)")

In [ ]:
from src.verify import score_predictions_file

print("=== Oracle baseline ===")
oracle_report = score_predictions_file(
    rows_path, oracle_path, DATA / "report_oracle.json")

print()
print("=== Broken baseline ===")
broken_report = score_predictions_file(
    rows_path, broken_path, DATA / "report_broken.json")

Notice what the broken baseline reveals:

- **SACR ≈ 0.73** — about 27% of fields got corrupted (matches our injection rate).
- **SilentCriticalErrorRate ≈ 0.27** — the agent acted, corrupted, and reported success. This is the worst class of failure: the user thinks it worked.
- **PolicyComplianceRate ≈ 0.05** — almost no rows had the right procedure followed.
- **UnsafeCommitRate = 1.0** — every time policy required asking, the agent skipped.
- **CounterfactualActionSensitivity ≈ 0.82** — counterfactual pairs (one input changed) didn't always produce the expected one output change.

These are five **independent failure modes**. A flat scoring scheme would conflate them. Real voice agents can have high SACR but terrible PCR (rude and unsafe but accurate), or high PCR but low SACR (polite but wrong).

## 7. Hardcase mining — the *living* part of the benchmark

The biggest design choice in v1.0: this isn't a static dataset. The miner clusters baseline failures by `(invariant_type, ambiguity_class, task_type)`, looks up recipes for each cluster, and generates harder variants.

Equally important: when a failure cluster has *no recipe registered*, the miner surfaces that explicitly. **Coverage debt is visible, not hidden.**

In [ ]:
from src.hardcase_miner import mine_hardcases

summary = mine_hardcases(
    rows_path, DATA / "report_broken.json",
    DATA / "hardcases.jsonl",
    variants_per_pattern=10,
)

print(f"Mined {summary['n_hardcases']} hardcases across "
      f"{len(summary['patterns'])} failure patterns")
print()
print("Pattern coverage:")
for p in summary["patterns"]:
    status = p.get("status", "ok")
    marker = "✓" if status == "ok" else "✗"
    print(f"  {marker} {p['pattern']:55s} "
          f"n={p['n_examples']:3d} variants={p.get('n_variants_made', 0):3d}  ({status})")

The patterns with `no_recipe_registered` are the next thing to work on — they're cases where the broken baseline failed but the miner has no idea how to make a harder variant. To add coverage you write a new recipe in `src/hardcase_miner.py` and register it in `PATTERN_TO_RECIPES`. Each recipe is small and specific (decimal_shift_press, phonetic_teens_tens, tight_followup, buried_correction, spell_pressure). Keep them small; if a recipe is doing too many things, split it.

## 8. Contamination protection

A benchmark with a hidden eval split is only useful if submissions can't have trained on it. Two protections:

1. **Canary phrases.** A few hidden rows carry an unusual phrase (`CANARY-7F3A-VIB`) that should never appear in public splits. If a model's outputs reveal recognition of the canary, the submitter trained on the hidden split.
2. **SHA fingerprint overlap detection.** Hash (dialogue text + canonical values). Any matching fingerprints between hidden and public splits is contamination.

In [ ]:
from src.contamination_checks import find_overlap, assert_no_overlap

# Honest case: hidden and public splits derived from disjoint indices
public = DATA / "public.jsonl"
hidden = DATA / "hidden.jsonl"
with public.open("w") as f:
    for r in rows[:50]:
        f.write(r.model_dump_json() + "\n")
with hidden.open("w") as f:
    for r in rows[100:150]:
        f.write(r.model_dump_json() + "\n")
assert_no_overlap(hidden, [public])

# Adversarial case: someone trying to sneak public rows into the hidden split
bad_hidden = DATA / "bad_hidden.jsonl"
with bad_hidden.open("w") as f:
    for r in rows[:5]:    # these ARE in public
        f.write(r.model_dump_json() + "\n")

overlaps = find_overlap(bad_hidden, [public])
print(f"\nDetected {len(overlaps)} planted overlaps:")
for h, p in overlaps:
    print(f"  hidden {h} == public {p}")

## 9. Run the test suite

11 tests covering canonicalization roundtrips, lineage recording, counterfactual integrity, policy decisions in three regimes, oracle scoring 100%, and the broken baseline producing nonzero SCER.

In [ ]:
!PYTHONPATH=. python -m pytest tests/ -v 2>&1 | tail -25

## 10. Optional: TTS + ASR (needs GPU runtime)

The cells in this section are commented out. To run them:

1. Switch to a GPU runtime: *Runtime → Change runtime type → T4 GPU (or better)*.
2. Uncomment and run the install cell.
3. Uncomment and run the synthesis + ASR cells.

On a T4, TTS for 50 turns takes ~2-3 minutes; faster-whisper ASR is near real-time. For the full Phase 0 MVP (~3k rows) you want an A100 80GB on RunPod — see `scripts/run_mvp.sh` and `COSTS.md`.

In [ ]:
# # GPU install — only enable on a GPU runtime
# !pip install -q TTS==0.22.0 faster-whisper torchaudio audiomentations soundfile

In [ ]:
# # Synthesize audio for the first 20 rows
# from pathlib import Path
# from src.tts_synth import synthesize_jsonl
#
# small = DATA / "rows_small.jsonl"
# with rows_path.open() as fin, small.open("w") as fout:
#     for i, line in enumerate(fin):
#         if i >= 20: break
#         fout.write(line)
#
# synthesize_jsonl(small, DATA / "rows_audio.jsonl",
#                  DATA / "audio", seed=0, dry_run=False)

In [ ]:
# # Listen to one synthesized turn
# import IPython.display as ipd
# import json
# row = json.loads((DATA / "rows_audio.jsonl").open().readline())
# first_audio = DATA / "audio" / row["audio_dialogue"][0]["audio"]
# print(f"Row: {row['id']}")
# print(f"Text: {row['audio_dialogue'][0]['text']}")
# ipd.Audio(str(first_audio))

In [ ]:
# # ASR audit
# from src.asr_audit import audit_jsonl
# audit_jsonl(DATA / "rows_audio.jsonl", DATA / "audio",
#             DATA / "rows_audited.jsonl")
#
# # Inspect ASR recovery
# import json
# for i, line in enumerate((DATA / "rows_audited.jsonl").open()):
#     if i >= 5: break
#     row = json.loads(line)
#     audit = row.get("asr_audit", {})
#     print(f"transcript: {audit.get('user_transcript', '')[:120]}")
#     for r in audit.get("invariant_recovery", []):
#         print(f"  {r['type']}: expected={r['expected_canonical']!r} "
#               f"recovered={r['recovered_canonical']!r} match={r['match']}")
#     print()

## 11. Using your own agent

Two paths.

**Option A — Offline predictions.** Run your agent however you want on `rows.jsonl`, write a predictions file with the same row IDs in this shape:

```json
{"id": "vib_...", "tool_calls": [...], "agent_messages": [...],
 "final_state": {...}, "final_response": "..."}
```

Then:

```bash
python eval/score_predictions.py score \
    --dataset data/splits/benchmark_core.jsonl \
    --predictions your_preds.jsonl \
    --output report.json
```

**Option B — Live HTTP endpoint.** If your agent is a service:

```bash
python eval/score_predictions.py run \
    --dataset data/splits/benchmark_core.jsonl \
    --agent_endpoint http://localhost:8000/chat \
    --output report.json
```

The evaluator POSTs each row to your endpoint and scores the response. Your endpoint should accept `{"id", "dialogue", "audio_dialogue", "tools", "initial_state"}` and return the predictions shape above.

In [ ]:
# Demonstrate scoring against the CLI exactly as a user would
!PYTHONPATH=. python eval/score_predictions.py score \
    --dataset /tmp/vib_demo/rows.jsonl \
    --predictions /tmp/vib_demo/preds_broken.jsonl \
    --output /tmp/vib_demo/cli_report.json

## 12. What's next

You've now run every architectural piece. The full pipeline for a real release is in `scripts/`:

- **`scripts/setup_runpod.sh`** — one-shot setup of a fresh RunPod A100/H100. Pre-downloads XTTS-v2 and faster-whisper.
- **`scripts/run_mvp.sh`** — Phase 0 (~3k rows, ~$45-70, 1-3 days).
- **`scripts/run_phase1.sh`** — Phase 1 (~8k rows, sharded, ~$175-260, 2-4 weeks).

See `ROADMAP.md` for the phased plan (the original 1M-row target is dropped — quality at 5-10k beats volume at 1M) and `COSTS.md` for verified RunPod pricing.

### Where to extend

- **Add a scenario.** Write invariant factories in `src/scenarios.py`, add a tool schema in `tools/`. About 30 lines for a new domain.
- **Add a hardcase recipe.** New small mutation in `src/hardcase_miner.py`, register it in `PATTERN_TO_RECIPES`. Watch the miner stop saying `no_recipe_registered` for that pattern.
- **Add a policy rule.** New `PolicyRule` in `DEFAULT_RULES` in `src/policy_engine.py`. Re-run the smoke test to see PCR / UCR shift.
- **Add a renderer language.** Subclass `DialogueRenderer`, swap the template banks. The planner doesn't change.

### Citation / contact

This is the v1.0 architecture described in the README. If you build on it, the `provenance` field on every row tracks the code version, schema version, and seeds — so reproducing or auditing any specific row is one lookup away.